# SafeFundus — Notebook 06C: Swin-Tiny ASL Normal-Aware Multi-Seed Classifier

## Why this notebook exists

The first JointEnergy experiment exposed a structural problem:

- familiar **known-disease** images usually produce positive known-disease logits;
- familiar **normal** images have all 14 disease targets equal to zero;
- unknown rare images also have no true known-disease target.

Because the original classifier had only the 14 disease outputs, JointEnergy often treated normal images as unfamiliar.

To correct this, this notebook retrains the already-selected **Swin-Tiny + Asymmetric Loss (ASL)** classifier with an explicit **NORMAL** output.

## New 15-output target space

\[
[\text{NORMAL}, \text{DR}, \text{MH}, \ldots, \text{CRS}]
\]

For a normal image:

\[
NORMAL=1,\qquad \text{all 14 disease labels}=0
\]

For a known-disease image:

\[
NORMAL=0,\qquad \text{one or more known disease labels}=1
\]

The disease task remains multi-label.

## Seeds

- 42
- 123
- 2026

## What remains frozen

- RFMiD split protocol
- 14 known diseases
- unknown/intermediate exclusions from training
- Swin-Tiny backbone
- ImageNet initialization
- ASL hyperparameters
- 224 × 224 input
- optimizer and scheduler
- validation-mAP model selection
- validation-only threshold optimization
- no unknown-test tuning

## Important

The main disease metrics are still calculated on the original **14 disease outputs**.

The additional NORMAL output is evaluated separately and is primarily introduced to make the later open-set energy representation normal-aware.


In [1]:
# ============================================================
# 1. Imports
# ============================================================

from pathlib import Path
import os
import json
import random
import time
import warnings
import shutil
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import Swin_T_Weights
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


# 2. Configuration


In [2]:
# ============================================================
# 2. Configuration
# ============================================================

CONFIG = {
    "seeds": [42, 123, 2026],

    "image_size": 224,
    "batch_size": 32,
    "epochs": 35,

    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "dropout": 0.30,

    "num_workers": 2,
    "early_stopping_patience": 7,
    "lr_patience": 3,

    "threshold_grid_start": 0.05,
    "threshold_grid_end": 0.95,
    "threshold_grid_step": 0.01,

    "asl_gamma_neg": 4.0,
    "asl_gamma_pos": 1.0,
    "asl_clip": 0.05,

    "model_name": "swin_tiny",
    "loss_name": "asymmetric_loss",
    "normal_aware": True,
    "num_outputs": 15,
}

DATA_ROOT = Path(
    "/kaggle/input/datasets/ozlemhakdagli/"
    "retinal-fundus-multi-disease-image-dataset-rfmid"
)

TRAIN_DIR = DATA_ROOT / "Training_set"
VAL_DIR   = DATA_ROOT / "Validation_set"
TEST_DIR  = DATA_ROOT / "Test_set"

TRAIN_CSV = TRAIN_DIR / "RFMiD_Training_Labels.csv"
VAL_CSV   = VAL_DIR   / "RFMiD_Validation_Labels.csv"
TEST_CSV  = TEST_DIR  / "RFMiD_Testing_Labels.csv"

NOTEBOOK_STEM = "06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier"

OUTPUT_DIR = Path("/kaggle/working") / NOTEBOOK_STEM
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"

for p in [
    OUTPUT_DIR,
    CHECKPOINT_DIR,
    TABLE_DIR,
    FIG_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(json.dumps(CONFIG, indent=2))
print("\nDEVICE:", DEVICE)
print("OUTPUT:", OUTPUT_DIR)


{
  "seeds": [
    42,
    123,
    2026
  ],
  "image_size": 224,
  "batch_size": 32,
  "epochs": 35,
  "learning_rate": 0.0001,
  "weight_decay": 0.0001,
  "dropout": 0.3,
  "num_workers": 2,
  "early_stopping_patience": 7,
  "lr_patience": 3,
  "threshold_grid_start": 0.05,
  "threshold_grid_end": 0.95,
  "threshold_grid_step": 0.01,
  "asl_gamma_neg": 4.0,
  "asl_gamma_pos": 1.0,
  "asl_clip": 0.05,
  "model_name": "swin_tiny",
  "loss_name": "asymmetric_loss",
  "normal_aware": true,
  "num_outputs": 15
}

DEVICE: cuda
OUTPUT: /kaggle/working/06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier


# 3. Frozen Label Protocol


In [3]:
# ============================================================
# 3. Label protocol
# ============================================================

KNOWN_DISEASES = [
    "DR", "MH", "ODC", "TSLN", "DN", "MYA", "ARMD",
    "BRVO", "ODP", "ODE", "LS", "RS", "CSR", "CRS"
]

NORMAL_CLASS = "NORMAL"

MODEL_CLASSES = [
    NORMAL_CLASS,
    *KNOWN_DISEASES,
]

INTERMEDIATE_CLASSES = [
    "CRVO",
    "RPEC",
]

assert len(KNOWN_DISEASES) == 14
assert len(MODEL_CLASSES) == 15

print("Disease outputs:", len(KNOWN_DISEASES))
print("Total model outputs:", len(MODEL_CLASSES))
print(MODEL_CLASSES)


Disease outputs: 14
Total model outputs: 15
['NORMAL', 'DR', 'MH', 'ODC', 'TSLN', 'DN', 'MYA', 'ARMD', 'BRVO', 'ODP', 'ODE', 'LS', 'RS', 'CSR', 'CRS']


# 4. Load RFMiD and Reconstruct the Frozen Groups


In [4]:
# ============================================================
# 4. Load labels
# ============================================================

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

ID_COL = "ID"
RISK_COL = "Disease_Risk"

DISEASE_COLS = [
    c for c in train_df.columns
    if c not in {ID_COL, RISK_COL}
]

UNKNOWN_CLASSES = [
    c for c in DISEASE_COLS
    if c not in KNOWN_DISEASES
    and c not in INTERMEDIATE_CLASSES
]

def assign_group(row):
    positive = {
        c for c in DISEASE_COLS
        if int(row[c]) == 1
    }

    if len(positive) == 0:
        return "Normal"

    if positive & set(INTERMEDIATE_CLASSES):
        return "Contains_Intermediate"

    has_known = bool(
        positive & set(KNOWN_DISEASES)
    )

    has_unknown = bool(
        positive & set(UNKNOWN_CLASSES)
    )

    if has_known and has_unknown:
        return "Mixed_Known_Unknown"

    if has_known:
        return "Known_Only"

    if has_unknown:
        return "Unknown_Only"

    return "Other"

for df in [train_df, val_df, test_df]:
    df["Group"] = df.apply(
        assign_group,
        axis=1,
    )

for name, df in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df),
]:
    print(f"\n{name}")
    display(
        df["Group"]
        .value_counts()
        .rename_axis("Group")
        .reset_index(name="Count")
    )



TRAIN


,Group,Count
0,Known_Only,1312
1,Normal,401
2,Mixed_Known_Unknown,82
3,Unknown_Only,75
4,Contains_Intermediate,50



VALIDATION


,Group,Count
0,Known_Only,426
1,Normal,134
2,Unknown_Only,34
3,Mixed_Known_Unknown,32
4,Contains_Intermediate,14



TEST


,Group,Count
0,Known_Only,437
1,Normal,134
2,Unknown_Only,32
3,Mixed_Known_Unknown,24
4,Contains_Intermediate,13


# 5. Frozen Experimental Splits

Training and validation contain only:

- Normal
- Known-only

Unknown-only, mixed, and intermediate-containing images remain completely excluded from classifier training.


In [5]:
# ============================================================
# 5. Experimental splits
# ============================================================

train_exp = train_df[
    train_df["Group"].isin(
        ["Normal", "Known_Only"]
    )
].copy()

val_exp = val_df[
    val_df["Group"].isin(
        ["Normal", "Known_Only"]
    )
].copy()

test_familiar = test_df[
    test_df["Group"].isin(
        ["Normal", "Known_Only"]
    )
].copy()

test_unknown = test_df[
    test_df["Group"] == "Unknown_Only"
].copy()

test_mixed = test_df[
    test_df["Group"] == "Mixed_Known_Unknown"
].copy()

test_intermediate = test_df[
    test_df["Group"] == "Contains_Intermediate"
].copy()

assert len(train_exp) == 1713
assert len(val_exp) == 560
assert len(test_familiar) == 571
assert len(test_unknown) == 32
assert len(test_mixed) == 24

print("Train             :", len(train_exp))
print("Validation        :", len(val_exp))
print("Familiar test     :", len(test_familiar))
print("Unknown-only test :", len(test_unknown))
print("Mixed test        :", len(test_mixed))
print("Intermediate test :", len(test_intermediate))

print("\n✓ Frozen SafeFundus protocol verified.")


Train             : 1713
Validation        : 560
Familiar test     : 571
Unknown-only test : 32
Mixed test        : 24
Intermediate test : 13

✓ Frozen SafeFundus protocol verified.


# 6. Construct the Explicit NORMAL Target

This target is deterministic:

\[
NORMAL = 1
\]

only when the image contains no RFMiD disease label at all.

For Known_Only training/validation images:

\[
NORMAL = 0
\]


In [6]:
# ============================================================
# 6. NORMAL target
# ============================================================

def add_normal_target(df):
    out = df.copy()

    out[NORMAL_CLASS] = (
        out["Group"]
        == "Normal"
    ).astype(int)

    return out

for name in [
    "train_exp",
    "val_exp",
    "test_familiar",
    "test_unknown",
    "test_mixed",
    "test_intermediate",
]:
    globals()[name] = add_normal_target(
        globals()[name]
    )

print(
    "Training NORMAL positives:",
    int(train_exp[NORMAL_CLASS].sum())
)

print(
    "Validation NORMAL positives:",
    int(val_exp[NORMAL_CLASS].sum())
)

print(
    "Familiar-test NORMAL positives:",
    int(test_familiar[NORMAL_CLASS].sum())
)

assert int(train_exp[NORMAL_CLASS].sum()) == 401
assert int(val_exp[NORMAL_CLASS].sum()) == 134
assert int(test_familiar[NORMAL_CLASS].sum()) == 134


Training NORMAL positives: 401
Validation NORMAL positives: 134
Familiar-test NORMAL positives: 134


# 7. Verify Image Paths


In [7]:
# ============================================================
# 7. Image paths
# ============================================================

def normalize_id(x):
    return str(int(float(x)))

def add_image_paths(df, folder):
    out = df.copy()

    out["image_path"] = out[ID_COL].apply(
        lambda x: str(
            folder /
            f"{normalize_id(x)}.png"
        )
    )

    missing = [
        p for p in out["image_path"]
        if not Path(p).exists()
    ]

    if missing:
        raise FileNotFoundError(
            f"{len(missing)} images missing. "
            f"Example: {missing[:3]}"
        )

    return out

train_exp = add_image_paths(train_exp, TRAIN_DIR)
val_exp = add_image_paths(val_exp, VAL_DIR)
test_familiar = add_image_paths(test_familiar, TEST_DIR)
test_unknown = add_image_paths(test_unknown, TEST_DIR)
test_mixed = add_image_paths(test_mixed, TEST_DIR)
test_intermediate = add_image_paths(test_intermediate, TEST_DIR)

print("All image paths verified.")


All image paths verified.


# 8. Image Preprocessing

Kept identical to Notebook 06:

- 224 × 224
- horizontal flip
- ±10° rotation
- mild color jitter
- ImageNet normalization


In [8]:
# ============================================================
# 8. Transforms
# ============================================================

weights = Swin_T_Weights.IMAGENET1K_V1

imagenet_mean = weights.transforms().mean
imagenet_std = weights.transforms().std

train_transform = transforms.Compose([
    transforms.Resize(
        (
            CONFIG["image_size"],
            CONFIG["image_size"],
        )
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.08,
        hue=0.02,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std,
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize(
        (
            CONFIG["image_size"],
            CONFIG["image_size"],
        )
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std,
    ),
])


# 9. Dataset and DataLoaders


In [9]:
# ============================================================
# 9. Dataset
# ============================================================

class RFMiDNormalAwareDataset(Dataset):
    def __init__(
        self,
        dataframe,
        transform=None,
    ):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        target_values = [
            float(row[NORMAL_CLASS]),
            *[
                float(row[c])
                for c in KNOWN_DISEASES
            ],
        ]

        target = torch.tensor(
            target_values,
            dtype=torch.float32,
        )

        return {
            "image": image,
            "target": target,
            "id": normalize_id(row[ID_COL]),
            "group": row["Group"],
        }


In [10]:
# ============================================================
# 10. Reproducibility / loader factory
# ============================================================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = (
        torch.initial_seed()
        % (2**32)
    )
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loaders(seed):
    datasets = {
        "train": RFMiDNormalAwareDataset(
            train_exp,
            train_transform,
        ),
        "validation": RFMiDNormalAwareDataset(
            val_exp,
            eval_transform,
        ),
        "familiar": RFMiDNormalAwareDataset(
            test_familiar,
            eval_transform,
        ),
        "unknown": RFMiDNormalAwareDataset(
            test_unknown,
            eval_transform,
        ),
        "mixed": RFMiDNormalAwareDataset(
            test_mixed,
            eval_transform,
        ),
        "intermediate": RFMiDNormalAwareDataset(
            test_intermediate,
            eval_transform,
        ),
    }

    generator = torch.Generator()
    generator.manual_seed(seed)

    common = {
        "batch_size": CONFIG["batch_size"],
        "num_workers": CONFIG["num_workers"],
        "pin_memory": True,
        "persistent_workers": CONFIG["num_workers"] > 0,
    }

    loaders = {
        "train": DataLoader(
            datasets["train"],
            shuffle=True,
            worker_init_fn=seed_worker,
            generator=generator,
            **common,
        ),
    }

    for key in [
        "validation",
        "familiar",
        "unknown",
        "mixed",
        "intermediate",
    ]:
        loaders[key] = DataLoader(
            datasets[key],
            shuffle=False,
            **common,
        )

    return loaders


# 11. Asymmetric Loss

The ASL settings are identical to Notebook 06:

\[
\gamma_- = 4,\qquad
\gamma_+ = 1,\qquad
clip=0.05
\]


In [11]:
# ============================================================
# 11. ASL
# ============================================================

class AsymmetricLoss(nn.Module):
    def __init__(
        self,
        gamma_neg=4.0,
        gamma_pos=1.0,
        clip=0.05,
        eps=1e-8,
    ):
        super().__init__()

        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(
        self,
        logits,
        targets,
    ):
        prob_pos = torch.sigmoid(logits)
        prob_neg = 1.0 - prob_pos

        if self.clip is not None and self.clip > 0:
            prob_neg = (
                prob_neg + self.clip
            ).clamp(max=1.0)

        log_pos = torch.log(
            prob_pos.clamp(min=self.eps)
        )

        log_neg = torch.log(
            prob_neg.clamp(min=self.eps)
        )

        loss = (
            targets * log_pos
            +
            (1.0 - targets) * log_neg
        )

        if (
            self.gamma_neg > 0
            or self.gamma_pos > 0
        ):
            with torch.no_grad():
                pt = (
                    prob_pos * targets
                    +
                    prob_neg * (1.0 - targets)
                )

                gamma = (
                    self.gamma_pos * targets
                    +
                    self.gamma_neg * (1.0 - targets)
                )

                weight = torch.pow(
                    1.0 - pt,
                    gamma,
                )

            loss = loss * weight

        return -loss.mean()


criterion = AsymmetricLoss(
    gamma_neg=CONFIG["asl_gamma_neg"],
    gamma_pos=CONFIG["asl_gamma_pos"],
    clip=CONFIG["asl_clip"],
)

print(criterion)


AsymmetricLoss()


# 12. Normal-Aware Swin-Tiny

The classifier head now outputs **15 raw logits**:

\[
NORMAL + 14\text{ known diseases}
\]

The dropout remains available for the later MC Dropout experiment.


In [12]:
# ============================================================
# 12. Model
# ============================================================

def build_swin_tiny(
    num_outputs=15,
    dropout=0.30,
):
    try:
        model = models.swin_t(
            weights=Swin_T_Weights.IMAGENET1K_V1
        )

        print(
            "Loaded ImageNet pretrained "
            "Swin-Tiny weights."
        )

    except Exception as e:
        print(
            "WARNING: pretrained weights "
            "could not be loaded."
        )
        print("Reason:", e)

        model = models.swin_t(
            weights=None
        )

    in_features = model.head.in_features

    model.head = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(
            in_features,
            num_outputs,
        ),
    )

    return model


# 13. Metrics

The **model-selection metric remains disease validation mAP**, calculated only across the 14 disease outputs.

This is important: adding the NORMAL output does not change the primary disease-classification selection criterion.


In [13]:
# ============================================================
# 13. Metrics
# ============================================================

NORMAL_INDEX = 0
DISEASE_SLICE = slice(1, 15)

def safe_map(y_true, y_prob):
    values = []

    for c in range(y_true.shape[1]):
        if y_true[:, c].sum() == 0:
            continue

        values.append(
            average_precision_score(
                y_true[:, c],
                y_prob[:, c],
            )
        )

    return (
        float(np.mean(values))
        if values
        else np.nan
    )


def safe_macro_auroc(
    y_true,
    y_prob,
):
    values = []

    for c in range(y_true.shape[1]):
        if len(np.unique(y_true[:, c])) < 2:
            continue

        values.append(
            roc_auc_score(
                y_true[:, c],
                y_prob[:, c],
            )
        )

    return (
        float(np.mean(values))
        if values
        else np.nan
    )


def disease_epoch_metrics(
    y_true_15,
    y_prob_15,
    threshold=0.5,
):
    y_true = y_true_15[:, DISEASE_SLICE]
    y_prob = y_prob_15[:, DISEASE_SLICE]

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    return {
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "micro_f1": f1_score(
            y_true,
            y_pred,
            average="micro",
            zero_division=0,
        ),
        "mAP": safe_map(
            y_true,
            y_prob,
        ),
        "macro_auroc": safe_macro_auroc(
            y_true,
            y_prob,
        ),
    }


def normal_metrics(
    y_true_15,
    y_prob_15,
    threshold=0.5,
):
    yt = y_true_15[:, NORMAL_INDEX].astype(int)
    prob = y_prob_15[:, NORMAL_INDEX]
    yp = (prob >= threshold).astype(int)

    return {
        "normal_f1": f1_score(
            yt,
            yp,
            zero_division=0,
        ),
        "normal_precision": precision_score(
            yt,
            yp,
            zero_division=0,
        ),
        "normal_recall": recall_score(
            yt,
            yp,
            zero_division=0,
        ),
        "normal_auroc": (
            roc_auc_score(yt, prob)
            if len(np.unique(yt)) == 2
            else np.nan
        ),
        "normal_ap": (
            average_precision_score(yt, prob)
            if yt.sum() > 0
            else np.nan
        ),
    }


# 14. Training and Evaluation


In [14]:
# ============================================================
# 14. Train / evaluate functions
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
):
    model.train()

    running_loss = 0.0
    all_targets = []
    all_probs = []

    for batch in tqdm(
        loader,
        desc="Train",
        leave=False,
    ):
        images = batch["image"].to(
            DEVICE,
            non_blocking=True,
        )

        targets = batch["target"].to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.cuda.amp.autocast(
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model(images)
            loss = criterion(
                logits,
                targets,
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        probs = torch.sigmoid(logits)

        running_loss += (
            loss.item()
            * images.size(0)
        )

        all_targets.append(
            targets.detach().cpu().numpy()
        )

        all_probs.append(
            probs.detach().cpu().numpy()
        )

    y_true = np.concatenate(all_targets)
    y_prob = np.concatenate(all_probs)

    metrics = disease_epoch_metrics(
        y_true,
        y_prob,
    )

    metrics["loss"] = (
        running_loss
        / len(loader.dataset)
    )

    return metrics


@torch.no_grad()
def evaluate_loader(
    model,
    loader,
    calculate_loss=False,
):
    model.eval()

    running_loss = 0.0
    all_targets = []
    all_probs = []
    all_logits = []
    all_ids = []
    all_groups = []

    for batch in tqdm(
        loader,
        desc="Evaluate",
        leave=False,
    ):
        images = batch["image"].to(
            DEVICE,
            non_blocking=True,
        )

        targets = batch["target"].to(
            DEVICE,
            non_blocking=True,
        )

        with torch.cuda.amp.autocast(
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model(images)

            if calculate_loss:
                loss = criterion(
                    logits,
                    targets,
                )

                running_loss += (
                    loss.item()
                    * images.size(0)
                )

        probs = torch.sigmoid(logits)

        all_targets.append(
            targets.cpu().numpy()
        )
        all_probs.append(
            probs.cpu().numpy()
        )
        all_logits.append(
            logits.cpu().numpy()
        )
        all_ids.extend(batch["id"])
        all_groups.extend(batch["group"])

    y_true = np.concatenate(all_targets)
    y_prob = np.concatenate(all_probs)
    logits = np.concatenate(all_logits)

    disease_metrics = disease_epoch_metrics(
        y_true,
        y_prob,
    )

    result = {
        "metrics": disease_metrics,
        "targets": y_true,
        "probs": y_prob,
        "logits": logits,
        "ids": all_ids,
        "groups": all_groups,
    }

    if calculate_loss:
        result["metrics"]["loss"] = (
            running_loss
            / len(loader.dataset)
        )

    return result


# 15. Validation Threshold Optimization

We optimize:

- one NORMAL threshold;
- 14 disease thresholds;

all using validation data only.

The disease thresholds are used for familiar classification and safety analysis. The NORMAL threshold is reported separately and will be useful for later final-decision logic.


In [15]:
# ============================================================
# 15. Threshold optimization
# ============================================================

THRESHOLD_GRID = np.arange(
    CONFIG["threshold_grid_start"],
    CONFIG["threshold_grid_end"] + 1e-9,
    CONFIG["threshold_grid_step"],
)

def optimize_binary_threshold(
    y_true,
    y_prob,
):
    best_t = 0.5
    best_f1 = -1.0

    for threshold in THRESHOLD_GRID:
        pred = (
            y_prob >= threshold
        ).astype(int)

        score = f1_score(
            y_true,
            pred,
            zero_division=0,
        )

        if score > best_f1:
            best_f1 = score
            best_t = float(threshold)

    return best_t, best_f1


def optimize_all_thresholds(
    y_true_15,
    y_prob_15,
):
    thresholds = []
    rows = []

    for c, class_name in enumerate(
        MODEL_CLASSES
    ):
        threshold, best_f1 = (
            optimize_binary_threshold(
                y_true_15[:, c],
                y_prob_15[:, c],
            )
        )

        thresholds.append(threshold)

        rows.append({
            "Class": class_name,
            "Threshold": threshold,
            "Validation_F1": best_f1,
        })

    return (
        np.asarray(
            thresholds,
            dtype=np.float32,
        ),
        pd.DataFrame(rows),
    )


# 16. Per-Class Disease Metrics


In [16]:
# ============================================================
# 16. Per-class disease metrics
# ============================================================

def build_disease_metrics(
    y_true_15,
    y_prob_15,
    thresholds_15,
):
    y_true = y_true_15[:, DISEASE_SLICE]
    y_prob = y_prob_15[:, DISEASE_SLICE]
    thresholds = thresholds_15[DISEASE_SLICE]

    y_pred = (
        y_prob
        >= thresholds[None, :]
    ).astype(int)

    rows = []

    for c, disease in enumerate(
        KNOWN_DISEASES
    ):
        yt = y_true[:, c]
        yp = y_pred[:, c]
        prob = y_prob[:, c]

        tn, fp, fn, tp = confusion_matrix(
            yt,
            yp,
            labels=[0, 1],
        ).ravel()

        rows.append({
            "Disease": disease,
            "Positive_Test": int(yt.sum()),
            "Threshold": float(thresholds[c]),
            "Precision": precision_score(
                yt,
                yp,
                zero_division=0,
            ),
            "Sensitivity_Recall": recall_score(
                yt,
                yp,
                zero_division=0,
            ),
            "Specificity": (
                tn / (tn + fp)
                if (tn + fp) > 0
                else np.nan
            ),
            "F1": f1_score(
                yt,
                yp,
                zero_division=0,
            ),
            "AUROC": (
                roc_auc_score(yt, prob)
                if len(np.unique(yt)) == 2
                else np.nan
            ),
            "Average_Precision": (
                average_precision_score(yt, prob)
                if yt.sum() > 0
                else np.nan
            ),
            "TP": int(tp),
            "FP": int(fp),
            "TN": int(tn),
            "FN": int(fn),
        })

    return pd.DataFrame(rows)


# 17. Prediction Export Helper

Every saved file contains all **15 logits** and probabilities.

This is essential because the next JointEnergy notebook must operate on the full normal-aware logit vector.


In [17]:
# ============================================================
# 17. Export helper
# ============================================================

def prediction_dataframe(
    result,
    thresholds_15=None,
):
    df = pd.DataFrame({
        "ID": result["ids"],
        "Group": result["groups"],
    })

    for c, class_name in enumerate(
        MODEL_CLASSES
    ):
        df[
            f"true_{class_name}"
        ] = (
            result["targets"][:, c]
            .astype(int)
        )

        df[
            f"logit_{class_name}"
        ] = (
            result["logits"][:, c]
        )

        df[
            f"prob_{class_name}"
        ] = (
            result["probs"][:, c]
        )

        if thresholds_15 is not None:
            df[
                f"pred_{class_name}"
            ] = (
                result["probs"][:, c]
                >= thresholds_15[c]
            ).astype(int)

    return df


# 18. Multi-Seed Training

Model selection remains based on **14-disease validation mAP**, not on the NORMAL head and not on any unknown-test metric.


In [18]:
# ============================================================
# 18. Multi-seed training
# ============================================================

all_seed_summaries = []

for seed in CONFIG["seeds"]:

    print("\n" + "#" * 80)
    print(f"STARTING NORMAL-AWARE SEED {seed}")
    print("#" * 80)

    seed_everything(seed)
    loaders = make_loaders(seed)

    model = build_swin_tiny(
        num_outputs=len(MODEL_CLASSES),
        dropout=CONFIG["dropout"],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"],
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=CONFIG["lr_patience"],
        )
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=(DEVICE.type == "cuda")
    )

    seed_dir = OUTPUT_DIR / f"seed_{seed}"
    seed_table_dir = seed_dir / "tables"
    seed_pred_dir = seed_dir / "predictions"
    seed_fig_dir = seed_dir / "figures"

    for p in [
        seed_dir,
        seed_table_dir,
        seed_pred_dir,
        seed_fig_dir,
    ]:
        p.mkdir(parents=True, exist_ok=True)

    checkpoint_path = (
        CHECKPOINT_DIR /
        f"swin_tiny_asl_normalaware_seed_{seed}.pth"
    )

    history = []

    best_val_map = -np.inf
    best_epoch = None
    epochs_without_improvement = 0

    start_time = time.time()

    for epoch in range(
        1,
        CONFIG["epochs"] + 1,
    ):
        print(
            f"\nSeed {seed} | "
            f"Epoch {epoch}/{CONFIG['epochs']}"
        )

        train_metrics = train_one_epoch(
            model,
            loaders["train"],
            optimizer,
            scaler,
        )

        val_result = evaluate_loader(
            model,
            loaders["validation"],
            calculate_loss=True,
        )

        val_metrics = val_result["metrics"]

        scheduler.step(
            val_metrics["mAP"]
        )

        current_lr = (
            optimizer.param_groups[0]["lr"]
        )

        history.append({
            "seed": seed,
            "epoch": epoch,
            "lr": current_lr,

            "train_loss": train_metrics["loss"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_micro_f1": train_metrics["micro_f1"],
            "train_mAP": train_metrics["mAP"],
            "train_macro_auroc": train_metrics["macro_auroc"],

            "val_loss": val_metrics["loss"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_micro_f1": val_metrics["micro_f1"],
            "val_mAP": val_metrics["mAP"],
            "val_macro_auroc": val_metrics["macro_auroc"],
        })

        print(
            f"Train disease mAP={train_metrics['mAP']:.4f} | "
            f"Val disease mAP={val_metrics['mAP']:.4f} | "
            f"Val Macro-F1={val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["mAP"] > best_val_map:
            best_val_map = float(
                val_metrics["mAP"]
            )
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save({
                "model_state_dict": model.state_dict(),
                "seed": seed,
                "config": CONFIG,
                "model_classes": MODEL_CLASSES,
                "known_diseases": KNOWN_DISEASES,
                "best_val_disease_map": best_val_map,
                "best_epoch": best_epoch,
            }, checkpoint_path)

            print(
                "✓ Best checkpoint updated: "
                f"val disease mAP={best_val_map:.4f}"
            )

        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= CONFIG["early_stopping_patience"]
        ):
            print("Early stopping.")
            break

    training_minutes = (
        time.time()
        - start_time
    ) / 60.0

    history_df = pd.DataFrame(history)

    history_df.to_csv(
        seed_table_dir /
        "training_history.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Reload best checkpoint
    # --------------------------------------------------------

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model.eval()

    # --------------------------------------------------------
    # Validation thresholds
    # --------------------------------------------------------

    validation = evaluate_loader(
        model,
        loaders["validation"],
        calculate_loss=False,
    )

    thresholds_15, threshold_df = (
        optimize_all_thresholds(
            validation["targets"],
            validation["probs"],
        )
    )

    threshold_df.to_csv(
        seed_table_dir /
        "validation_thresholds_15class.csv",
        index=False,
    )

    prediction_dataframe(
        validation,
        thresholds_15,
    ).to_csv(
        seed_pred_dir /
        "validation_logits_predictions_15class.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Evaluate all test groups
    # --------------------------------------------------------

    results = {}

    for split_name in [
        "familiar",
        "unknown",
        "mixed",
        "intermediate",
    ]:
        results[split_name] = evaluate_loader(
            model,
            loaders[split_name],
            calculate_loss=False,
        )

        prediction_dataframe(
            results[split_name],
            thresholds_15,
        ).to_csv(
            seed_pred_dir /
            f"{split_name}_test_logits_predictions_15class.csv",
            index=False,
        )

    # --------------------------------------------------------
    # Familiar disease metrics
    # --------------------------------------------------------

    familiar = results["familiar"]

    disease_true = (
        familiar["targets"][:, DISEASE_SLICE]
    )
    disease_prob = (
        familiar["probs"][:, DISEASE_SLICE]
    )
    disease_thresholds = (
        thresholds_15[DISEASE_SLICE]
    )

    disease_pred = (
        disease_prob
        >= disease_thresholds[None, :]
    ).astype(int)

    familiar_macro_f1 = float(
        f1_score(
            disease_true,
            disease_pred,
            average="macro",
            zero_division=0,
        )
    )

    familiar_micro_f1 = float(
        f1_score(
            disease_true,
            disease_pred,
            average="micro",
            zero_division=0,
        )
    )

    familiar_map = safe_map(
        disease_true,
        disease_prob,
    )

    familiar_auroc = safe_macro_auroc(
        disease_true,
        disease_prob,
    )

    per_class_df = build_disease_metrics(
        familiar["targets"],
        familiar["probs"],
        thresholds_15,
    )

    per_class_df.to_csv(
        seed_table_dir /
        "per_class_disease_metrics.csv",
        index=False,
    )

    # --------------------------------------------------------
    # NORMAL head metrics on familiar test
    # --------------------------------------------------------

    normal_threshold = float(
        thresholds_15[NORMAL_INDEX]
    )

    familiar_normal_metrics = normal_metrics(
        familiar["targets"],
        familiar["probs"],
        threshold=normal_threshold,
    )

    # --------------------------------------------------------
    # Normal-vs-known decision consistency
    # --------------------------------------------------------

    normal_prob = (
        familiar["probs"][:, NORMAL_INDEX]
    )

    normal_pred = (
        normal_prob
        >= normal_threshold
    )

    familiar_groups = np.asarray(
        familiar["groups"]
    )

    true_normal_mask = (
        familiar_groups == "Normal"
    )

    true_known_mask = (
        familiar_groups == "Known_Only"
    )

    normal_sensitivity = float(
        normal_pred[
            true_normal_mask
        ].mean()
    )

    normal_false_positive_on_known = float(
        normal_pred[
            true_known_mask
        ].mean()
    )

    # --------------------------------------------------------
    # Closed-set safety on unknown-only cases
    # Disease decisions remain based on the 14 disease outputs.
    # --------------------------------------------------------

    unknown = results["unknown"]

    unknown_disease_prob = (
        unknown["probs"][:, DISEASE_SLICE]
    )

    unknown_disease_pred = (
        unknown_disease_prob
        >= disease_thresholds[None, :]
    ).astype(int)

    baseline_rare_to_normal = (
        unknown_disease_pred.sum(axis=1)
        == 0
    )

    rare_to_normal_count = int(
        baseline_rare_to_normal.sum()
    )

    rare_to_normal_rate = float(
        baseline_rare_to_normal.mean()
    )

    unknown_normal_prob_mean = float(
        unknown["probs"][:, NORMAL_INDEX].mean()
    )

    # --------------------------------------------------------
    # Generalization
    # --------------------------------------------------------

    best_history_row = history_df.loc[
        history_df["val_mAP"].idxmax()
    ]

    train_map_at_best = float(
        best_history_row["train_mAP"]
    )

    val_map_at_best = float(
        best_history_row["val_mAP"]
    )

    generalization_gap = float(
        train_map_at_best
        - val_map_at_best
    )

    seed_summary = {
        "seed": seed,
        "best_epoch": int(best_epoch),
        "training_minutes": float(training_minutes),

        "best_validation_disease_mAP": float(best_val_map),
        "train_disease_mAP_at_best_val": train_map_at_best,
        "disease_mAP_generalization_gap": generalization_gap,

        "familiar_test_images": int(len(familiar["ids"])),
        "disease_macro_f1": familiar_macro_f1,
        "disease_micro_f1": familiar_micro_f1,
        "disease_mAP": familiar_map,
        "disease_macro_auroc": familiar_auroc,

        "normal_threshold": normal_threshold,
        **{
            k: float(v)
            for k, v in familiar_normal_metrics.items()
        },

        "normal_sensitivity_on_true_normal":
            normal_sensitivity,
        "normal_false_positive_rate_on_known":
            normal_false_positive_on_known,

        "unknown_only_test_images":
            int(len(unknown["ids"])),
        "closedset_rare_to_normal_count":
            rare_to_normal_count,
        "closedset_rare_to_normal_error_rate":
            rare_to_normal_rate,
        "unknown_mean_normal_probability":
            unknown_normal_prob_mean,
    }

    all_seed_summaries.append(
        seed_summary
    )

    with open(
        seed_table_dir /
        "seed_summary.json",
        "w",
    ) as f:
        json.dump(
            seed_summary,
            f,
            indent=2,
        )

    # --------------------------------------------------------
    # Curves
    # --------------------------------------------------------

    plt.figure(figsize=(8, 5))

    plt.plot(
        history_df["epoch"],
        history_df["train_mAP"],
        label="Train disease mAP",
    )

    plt.plot(
        history_df["epoch"],
        history_df["val_mAP"],
        label="Validation disease mAP",
    )

    plt.xlabel("Epoch")
    plt.ylabel("Disease mAP")
    plt.title(
        f"Normal-Aware Swin-Tiny ASL — Seed {seed}"
    )
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        seed_fig_dir /
        "disease_map_curve.png",
        dpi=200,
        bbox_inches="tight",
    )

    plt.close()

    print("\nSeed summary:")
    print(
        json.dumps(
            seed_summary,
            indent=2,
        )
    )

    del model
    del optimizer
    del scheduler
    del scaler
    del loaders

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nAll three normal-aware seeds completed.")



################################################################################
STARTING NORMAL-AWARE SEED 42
################################################################################
Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth


100%|██████████| 108M/108M [00:00<00:00, 206MB/s] 


Loaded ImageNet pretrained Swin-Tiny weights.

Seed 42 | Epoch 1/35


Train:   0%|          | 0/54 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/18 [00:00<?, ?it/s]

Train disease mAP=0.2014 | Val disease mAP=0.4083 | Val Macro-F1=0.2814
✓ Best checkpoint updated: val disease mAP=0.4083

Seed 42 | Epoch 2/35


... (221 outputs left out)

Evaluate:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/1 [00:00<?, ?it/s]


Seed summary:
{
  "seed": 2026,
  "best_epoch": 18,
  "training_minutes": 71.35031818151474,
  "best_validation_disease_mAP": 0.7094059303795092,
  "train_disease_mAP_at_best_val": 0.9660014101224642,
  "disease_mAP_generalization_gap": 0.25659547974295505,
  "familiar_test_images": 571,
  "disease_macro_f1": 0.6561110522796005,
  "disease_micro_f1": 0.7072727272727273,
  "disease_mAP": 0.7355699428087762,
  "disease_macro_auroc": 0.9401677676211799,
  "normal_threshold": 0.3799999952316284,
  "normal_f1": 0.7638483965014577,
  "normal_precision": 0.6267942583732058,
  "normal_recall": 0.9776119402985075,
  "normal_auroc": 0.9680402336145361,
  "normal_ap": 0.8880228379309624,
  "normal_sensitivity_on_true_normal": 0.9776119402985075,
  "normal_false_positive_rate_on_known": 0.17848970251716248,
  "unknown_only_test_images": 32,
  "closedset_rare_to_normal_count": 10,
  "closedset_rare_to_normal_error_rate": 0.3125,
  "unknown_mean_normal_probability": 0.285888671875
}

All three norm

# 19. Aggregate Three-Seed Results

The main question here is whether adding NORMAL preserves acceptable disease-classification performance while providing a useful explicit familiar-normal signal.


In [19]:
# ============================================================
# 19. Aggregate
# ============================================================

seed_results_df = pd.DataFrame(
    all_seed_summaries
)

seed_results_df.to_csv(
    TABLE_DIR /
    "all_seed_results.csv",
    index=False,
)

display(seed_results_df)

AGGREGATE_COLUMNS = [
    "best_validation_disease_mAP",
    "disease_mAP_generalization_gap",
    "disease_macro_f1",
    "disease_micro_f1",
    "disease_mAP",
    "disease_macro_auroc",
    "normal_f1",
    "normal_auroc",
    "normal_sensitivity_on_true_normal",
    "normal_false_positive_rate_on_known",
    "closedset_rare_to_normal_error_rate",
    "unknown_mean_normal_probability",
]

aggregate_rows = []

for metric in AGGREGATE_COLUMNS:
    values = pd.to_numeric(
        seed_results_df[metric],
        errors="coerce",
    )

    aggregate_rows.append({
        "Metric": metric,
        "Mean": float(values.mean()),
        "SD": float(values.std(ddof=1)),
        "Min": float(values.min()),
        "Max": float(values.max()),
    })

aggregate_df = pd.DataFrame(
    aggregate_rows
)

aggregate_df.to_csv(
    TABLE_DIR /
    "multiseed_mean_sd.csv",
    index=False,
)

display(aggregate_df)

print("\n" + "=" * 80)
print(
    "SWIN-TINY ASL NORMAL-AWARE MULTI-SEED SUMMARY"
)
print("=" * 80)

for _, row in aggregate_df.iterrows():
    print(
        f"{row['Metric']:45s}: "
        f"{row['Mean']:.6f} ± {row['SD']:.6f}"
    )


,seed,best_epoch,training_minutes,best_validation_disease_mAP,train_disease_mAP_at_best_val,disease_mAP_generalization_gap,familiar_test_images,disease_macro_f1,disease_micro_f1,disease_mAP,...,normal_precision,normal_recall,normal_auroc,normal_ap,normal_sensitivity_on_true_normal,normal_false_positive_rate_on_known,unknown_only_test_images,closedset_rare_to_normal_count,closedset_rare_to_normal_error_rate,unknown_mean_normal_probability
0,42,10,48.708003,0.727051,0.864732,0.137681,571,0.667647,0.690843,0.736494,...,0.711957,0.977612,0.976835,0.918979,0.977612,0.121281,32,7,0.21875,0.314453
1,123,21,79.186958,0.727633,0.982041,0.254408,571,0.671954,0.701181,0.735915,...,0.664975,0.977612,0.974538,0.921706,0.977612,0.151030,32,9,0.28125,0.227539
2,2026,18,71.350318,0.709406,0.966001,0.256595,571,0.656111,0.707273,0.735570,...,0.626794,0.977612,0.968040,0.888023,0.977612,0.178490,32,10,0.31250,0.285889


,Metric,Mean,SD,Min,Max
0,best_validation_disease_mAP,0.721363,0.010359,0.709406,0.727633
1,disease_mAP_generalization_gap,0.216228,0.068033,0.137681,0.256595
2,disease_macro_f1,0.665237,0.008192,0.656111,0.671954
3,disease_micro_f1,0.699766,0.008306,0.690843,0.707273
4,disease_mAP,0.735993,0.000467,0.735570,0.736494
5,disease_macro_auroc,0.942239,0.008472,0.934994,0.951554
6,normal_f1,0.793096,0.030056,0.763848,0.823899
7,normal_auroc,0.973138,0.004562,0.968040,0.976835
8,normal_sensitivity_on_true_normal,0.977612,0.000000,0.977612,0.977612
9,normal_false_positive_rate_on_known,0.150267,0.028612,0.121281,0.178490



SWIN-TINY ASL NORMAL-AWARE MULTI-SEED SUMMARY
best_validation_disease_mAP                  : 0.721363 ± 0.010359
disease_mAP_generalization_gap               : 0.216228 ± 0.068033
disease_macro_f1                             : 0.665237 ± 0.008192
disease_micro_f1                             : 0.699766 ± 0.008306
disease_mAP                                  : 0.735993 ± 0.000467
disease_macro_auroc                          : 0.942239 ± 0.008472
normal_f1                                    : 0.793096 ± 0.030056
normal_auroc                                 : 0.973138 ± 0.004562
normal_sensitivity_on_true_normal            : 0.977612 ± 0.000000
normal_false_positive_rate_on_known          : 0.150267 ± 0.028612
closedset_rare_to_normal_error_rate          : 0.270833 ± 0.047735
unknown_mean_normal_probability              : 0.275960 ± 0.044299


# 20. Compare Against the Original 14-Output ASL Classifier

The original Notebook 06 ASL model had mean validation disease mAP:

\[
0.7323 \pm 0.0060
\]

This comparison is diagnostic.

The normal-aware model is acceptable if it provides a much better representation of normality without causing a severe drop in the original disease task.


In [20]:
# ============================================================
# 20. Diagnostic comparison
# ============================================================

ORIGINAL_ASL = {
    "best_validation_disease_mAP_mean":
        0.7323,
    "best_validation_disease_mAP_sd":
        0.0060,
}

normalaware_val_map_mean = float(
    seed_results_df[
        "best_validation_disease_mAP"
    ].mean()
)

normalaware_val_map_sd = float(
    seed_results_df[
        "best_validation_disease_mAP"
    ].std(ddof=1)
)

comparison = {
    "original_14output_ASL_val_mAP_mean":
        ORIGINAL_ASL[
            "best_validation_disease_mAP_mean"
        ],
    "original_14output_ASL_val_mAP_sd":
        ORIGINAL_ASL[
            "best_validation_disease_mAP_sd"
        ],
    "normalaware_15output_ASL_val_mAP_mean":
        normalaware_val_map_mean,
    "normalaware_15output_ASL_val_mAP_sd":
        normalaware_val_map_sd,
    "normalaware_minus_original":
        normalaware_val_map_mean
        - ORIGINAL_ASL[
            "best_validation_disease_mAP_mean"
        ],
}

print(
    json.dumps(
        comparison,
        indent=2,
    )
)

with open(
    TABLE_DIR /
    "normalaware_vs_original_asl.json",
    "w",
) as f:
    json.dump(
        comparison,
        f,
        indent=2,
    )


{
  "original_14output_ASL_val_mAP_mean": 0.7323,
  "original_14output_ASL_val_mAP_sd": 0.006,
  "normalaware_15output_ASL_val_mAP_mean": 0.7213631497063839,
  "normalaware_15output_ASL_val_mAP_sd": 0.010359341188979986,
  "normalaware_minus_original": -0.010936850293616018
}


# 21. Save Reproducibility Metadata


In [21]:
# ============================================================
# 21. Metadata
# ============================================================

metadata = {
    "project":
        "SafeFundus",

    "classifier":
        "Swin-Tiny + ASL + explicit NORMAL output",

    "model_classes":
        MODEL_CLASSES,

    "known_diseases":
        KNOWN_DISEASES,

    "normal_target_definition":
        "1 only for RFMiD images with no disease labels; otherwise 0",

    "seeds":
        CONFIG["seeds"],

    "model_selection_metric":
        "14-disease validation mAP",

    "unknown_test_used_for_model_selection":
        False,

    "next_method":
        "Normal-aware JointEnergy using all 15 logits",
}

with open(
    OUTPUT_DIR /
    "config.json",
    "w",
) as f:
    json.dump(
        CONFIG,
        f,
        indent=2,
    )

with open(
    OUTPUT_DIR /
    "normalaware_method_metadata.json",
    "w",
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
    )

print(
    json.dumps(
        metadata,
        indent=2,
    )
)


{
  "project": "SafeFundus",
  "classifier": "Swin-Tiny + ASL + explicit NORMAL output",
  "model_classes": [
    "NORMAL",
    "DR",
    "MH",
    "ODC",
    "TSLN",
    "DN",
    "MYA",
    "ARMD",
    "BRVO",
    "ODP",
    "ODE",
    "LS",
    "RS",
    "CSR",
    "CRS"
  ],
  "known_diseases": [
    "DR",
    "MH",
    "ODC",
    "TSLN",
    "DN",
    "MYA",
    "ARMD",
    "BRVO",
    "ODP",
    "ODE",
    "LS",
    "RS",
    "CSR",
    "CRS"
  ],
  "normal_target_definition": "1 only for RFMiD images with no disease labels; otherwise 0",
  "seeds": [
    42,
    123,
    2026
  ],
  "model_selection_metric": "14-disease validation mAP",
  "unknown_test_used_for_model_selection": false,
  "next_method": "Normal-aware JointEnergy using all 15 logits"
}


# 22. Create One Complete Output ZIP

The notebook automatically creates:

**`06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier.zip`**

The ZIP contains:

- all 3 best 15-output checkpoints;
- all training histories;
- all validation thresholds;
- validation logits;
- familiar-test logits;
- unknown-only logits;
- mixed known+unknown logits;
- intermediate-case logits;
- disease metrics;
- NORMAL-head metrics;
- mean ± SD tables;
- reproducibility metadata.

This single ZIP will be the input for the corrected **normal-aware JointEnergy** notebook.


In [22]:
# ============================================================
# 22. Create ZIP
# ============================================================

ZIP_PATH = (
    Path("/kaggle/working")
    / f"{NOTEBOOK_STEM}.zip"
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(
    base_name=str(
        ZIP_PATH.with_suffix("")
    ),
    format="zip",
    root_dir=OUTPUT_DIR,
)

zip_size_mb = (
    ZIP_PATH.stat().st_size
    / (1024 ** 2)
)

print("=" * 80)
print("COMPLETE OUTPUT ZIP CREATED")
print("=" * 80)
print(ZIP_PATH)
print(
    f"ZIP size: {zip_size_mb:.1f} MB"
)


COMPLETE OUTPUT ZIP CREATED
/kaggle/working/06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier.zip
ZIP size: 292.8 MB


# Next Step

Upload:

**`06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier.zip`**

I will first verify:

1. whether disease validation mAP remains acceptable;
2. whether the NORMAL output cleanly separates normal from known-disease familiar cases;
3. how strongly unknown-only cases activate the NORMAL output;
4. whether the new representation is suitable for normal-aware JointEnergy.

Then the next notebook will be:

## `07B_NormalAware_JointEnergy_OpenSet_Scoring.ipynb`

using all **15 logits** and **no retraining**.
